## Installing Python Packages

In [5]:
!pip install pylatexenc
!pip install qiskit
!pip install qiskit-aer
!pip install opencv-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 7.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136897 sha256=c2f2dfe99e56786271cfcb7e1ae626a5d606ea59863c493aefba51422447a885
  Stored in directory: /root/.cache/pip/wheels/b1/7a/33/9fdd892f784ed4afda62b685ae3703adf4c91aa0f524c28f03
Successfully built pylatexenc
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 118.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.0/109.0 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━

## Importing Python Packages

In [6]:
import os
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
from numpy import asarray
from pathlib import Path
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.circuit.library import RYGate
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import Permutation, UnitaryGate
from qiskit.circuit.library import MCXGate
from qiskit.quantum_info import Operator
import time
import math
from skimage.measure import shannon_entropy
from skimage.morphology import skeletonize

## Simulation Setup

In [3]:
backend_sim = AerSimulator(method="statevector")

def run_simulation_statevector(qc,backend_sim):
  compiled_circuit = transpile(qc, backend_sim,optimization_level=1)
  result = backend_sim.run(compiled_circuit).result()
  statevector = result.get_statevector(compiled_circuit)
  statevector = np.real(np.asarray(statevector))

  return statevector

## Image Pre-Processing

In [4]:
def load_and_display_image(image_path, display_img = True, invert = False):
    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    if invert:
        image = 255 - image
    if display_img:
        plt.imshow(image,cmap='gray')
        plt.title('Image')
        plt.axis('off')
        plt.show()

    return image

def resize_image(image, new_size=(64, 64)):
    resized_image = cv2.resize(image, new_size, interpolation=cv2.INTER_NEAREST)
    return resized_image

def crop_image(image, new_size=(64, 64)):
    croped_image = image[0:new_size[0], 0:new_size[1]]
    return croped_image

def save_image(image, file_name = None):
    plt.imsave(file_name, image, cmap='gray')

def plot_image(image, title):
    size_img = np.shape(image)
    title = title + " " + str(size_img)

    plt.imshow(image, extent=[0,image.shape[0], image.shape[1],0,], cmap='gray')
    plt.xticks(range(0,image.shape[0],2))
    plt.yticks(range(0,image.shape[1],2))
    plt.axis('off')
    plt.title(title)
    plt.show()

def plot_images(images, titles):
    size_img = np.shape(images[0])
    f, axarr = plt.subplots(1,len(images), figsize=(15,3*len(images)))

    for i in range(len(images)):
        title = titles[i] + " " + str(size_img)
        axarr[i].set_title(title)
        axarr[i].set_xticks(range(0,images[i].shape[0],10))
        axarr[i].set_yticks(range(0,images[i].shape[1],10))
        axarr[i].imshow(images[i], cmap='gray')
        axarr[i].axis('off')

    plt.show()

## Quantum Probability Image Encoding

In [5]:
def qpie(img, measure = True):
    img = img/np.linalg.norm(img)

    image_array=np.asarray(img).flatten()
    qubit_num = int(np.log2(len(image_array)))

    position_qubits = QuantumRegister(qubit_num, 'position')
    classical_bits = ClassicalRegister(qubit_num, 'classical')
    qc = QuantumCircuit(position_qubits, classical_bits)

    qc.initialize(image_array,range(qubit_num))
    if measure:
        qc.measure_all(add_bits=False)
        return qc
    else:
        return qc, position_qubits, classical_bits, qubit_num

def qpie_gradient(img, grad_type = None):
  qed_bit = ClassicalRegister(1, 'qed_bit')
  qed_qubit = QuantumRegister(1, "qed")

  qpie_qc, position_qubits, classical_bits , qubit_num = qpie(img, measure=False)

  # Build a new circuit with registers in the order: psi_reg, qed_qubit, then classical registers.
  qc = QuantumCircuit(qed_qubit,position_qubits,qed_bit,classical_bits)
  qc.compose(qpie_qc, qubits=position_qubits, inplace=True)

  if grad_type == 'lag1' or grad_type == 'lag2':
    qc.h(qed_qubit[0])
    #################################
    qc.barrier()
    ### Permutation operator
    qc.x(qed_qubit[0])
    for i in range(qubit_num):
        if i == 0:
            qc.cx(qed_qubit[0], position_qubits[0])
        else:
            # Define controls: always include qhed[0] and the first i position qubits.
            controls = [qed_qubit[0]] + list(position_qubits[:i])
            target = position_qubits[i]
            # Append an MCX gate with the appropriate number of controls.
            qc.append(MCXGate(len(controls)), qargs=controls + [target])
    #################################
    qc.barrier()
    #################################
    if grad_type == 'lag2':
      qc.x(qed_qubit[0])
      #################################
      qc.barrier()
      ### Permutation operator
      qc.x(qed_qubit[0])
      for i in range(qubit_num):
          if i == 0:
              qc.cx(qed_qubit[0], position_qubits[0])
          else:
              # Define controls: always include qhed[0] and the first i position qubits.
              controls = [qed_qubit[0]] + list(position_qubits[:i])
              target = position_qubits[i]
              # Append an MCX gate with the appropriate number of controls.
              qc.append(MCXGate(len(controls)), qargs=controls + [target])
      #################################
    #################################
    qc.barrier()
    qc.h(qed_qubit[0])
  else: print("Error! Please provide kernel type.")
  qc.save_statevector()

  return qc, grad_type

def retrieve_grad_qpie(statevector, grad_type = 'None', Sobel = False):
    n = int((np.log2(len(statevector))-1)//2) # Get n from the first dict key length

    ### grad = [c0-c2, c1-c3, c2-c4, ----]
    grad = [statevector[i] for i in range(1,len(statevector)+1,2)]


    if grad_type == 'lag1':
      grad = np.asarray(grad)
      img_grad = grad.reshape((2**n,2**n))

      return img_grad

    elif grad_type == 'lag2':
      img_grad = grad[-1:] + grad[:-1] ### Cyclic shift to the right

      img_grad = np.asarray(img_grad)
      img_grad = img_grad.reshape((2**n,2**n))

      #### construction of grad Sobel kernel
      if Sobel:
        sobel = np.zeros((2**n,2**n))
        for i in range(1,2**n-1):
          for j in range(1,2**n-1):
            sobel[i,j] = img_grad[i,j-1] + 2*img_grad[i,j] + img_grad[i,j+1]

        return sobel
      #######################################

      else: return img_grad

    else: print("Error! Please provide kernel type.")

## Flexible Representation of Quantum Images

In [6]:
def frqi(image, measure = True):
    image_array=np.asarray(image).flatten()
    qubit_num = int(np.log2(len(image_array)))

    pixel_qubit = QuantumRegister(1, 'pixel qubit')
    pos_qubits = QuantumRegister(qubit_num, 'position qubit')
    classical_bits = ClassicalRegister(qubit_num + 1, 'classical')
    qc = QuantumCircuit(pixel_qubit,pos_qubits, classical_bits)

    image_array = image_array / 255.0 ### Normalizing grayscale values
    # theta_values = [math.asin(image_array[k]) for k in range(len(image_array))]
    theta_values = [np.pi*image_array[k]/2 for k in range(len(image_array))]

    enc_state = [] ### Form: psi = cos(t)|i>|0> + sin(t)|i>|1>
    for k in range(0,2**(qubit_num+1),2):
      enc_state.append(np.cos(theta_values[int(k/2)]))
      enc_state.append(np.sin(theta_values[int(k/2)]))

    enc_state = np.asarray(enc_state)
    enc_state = enc_state/(2**(qubit_num/2))

    qc.initialize(enc_state,range(qubit_num+1))

    if measure:
        qc.measure_all(add_bits=False)
        return qc
    else:
        return qc, pos_qubits, pixel_qubit, classical_bits, qubit_num

def frqi_gradient(img, grad_type = None):

  qed_bit = ClassicalRegister(1, 'qhed_bit')

  qc, position_qubits, qed_qubit, classical_bits , qubit_num = frqi(img, measure=False)
  qc.add_register(qed_bit)

  # Build a new circuit with registers in the order: psi_reg, qed_qubit, then classical registers.
  # Measure qhed into classical bit c
  qc.measure(qed_qubit[0], qed_bit[0])

  # Conditionally apply X if measurement outcome == 1
  with qc.if_test((qed_bit[0], 1)):
      qc.x(qed_qubit[0])

  if grad_type == 'lag1' or grad_type == 'lag2':
    qc.h(qed_qubit[0])
    #################################
    qc.barrier()
    ### Permutation operator
    qc.x(qed_qubit[0])
    for i in range(qubit_num):
        if i == 0:
            qc.cx(qed_qubit[0], position_qubits[0])
        else:
            # Define controls: always include qhed[0] and the first i position qubits.
            controls = [qed_qubit[0]] + list(position_qubits[:i])
            target = position_qubits[i]
            # Append an MCX gate with the appropriate number of controls.
            qc.append(MCXGate(len(controls)), qargs=controls + [target])
    #################################
    qc.barrier()
    #################################
    if grad_type == 'lag2':
      qc.x(qed_qubit[0])
      #################################
      qc.barrier()
      ### Permutation operator
      qc.x(qed_qubit[0])
      for i in range(qubit_num):
          if i == 0:
              qc.cx(qed_qubit[0], position_qubits[0])
          else:
              # Define controls: always include qhed[0] and the first i position qubits.
              controls = [qed_qubit[0]] + list(position_qubits[:i])
              target = position_qubits[i]
              # Append an MCX gate with the appropriate number of controls.
              qc.append(MCXGate(len(controls)), qargs=controls + [target])
      #################################
    #################################
    qc.barrier()
    qc.h(qed_qubit[0])
  else: print("Error! Please provide kernel type.")
  qc.save_statevector()

  return qc, grad_type

def retrieve_grad_frqi(statevector, grad_type = 'None', Sobel = False):
    n = int((np.log2(len(statevector))-1)//2) # Get n from the first dict key length

    ### grad = [c0-c2, c1-c3, c2-c4, ----]
    grad = [statevector[i] for i in range(1,len(statevector)+1,2)]


    if grad_type == 'lag1':
      grad = np.asarray(grad)
      img_grad = grad.reshape((2**n,2**n))

      return img_grad

    elif grad_type == 'lag2':
      img_grad = grad[-1:] + grad[:-1] ### Cyclic shift to the right

      img_grad = np.asarray(img_grad)
      img_grad = img_grad.reshape((2**n,2**n))

      #### construction of grad Sobel kernel
      if Sobel:
        sobel = np.zeros((2**n,2**n))
        for i in range(1,2**n-1):
          for j in range(1,2**n-1):
            sobel[i,j] = img_grad[i,j-1] + 2*img_grad[i,j] + img_grad[i,j+1]

        return sobel
      #######################################

      else: return img_grad

    else: print("Error! Please provide kernel type.")

## Connecting to QHCD folder

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
### Urban Dataset
# dataset_path = Path('./drive/MyDrive/QHCD/UrbanDataset/Images')
# ### Animal images
dataset_path = Path('./drive/MyDrive/QHCD/Animals')
### Other images
# dataset_path = Path('./drive/MyDrive/QHCD/Edge Detection Img')

pixel_size = [256, 512, 1024]
pixel_size_num = len(pixel_size)

file_count = 0
for entry in dataset_path.iterdir():
      if entry.is_file() and entry.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']:
        print("Image File: ", entry.name)
        file_count = file_count + 1

print("\n Number of Test Images: ", file_count)

Image File:  EAGLE.webp
Image File:  LION.webp
Image File:  DEER.webp
Image File:  DOG.webp
Image File:  CAT.webp

 Number of Test Images:  5


## Edge Detection using QPIE

In [19]:
################################################
qpie_edge_density = np.zeros(file_count)
qpie_thickness_ratio = np.zeros(file_count)
qpie_edge_fragments = np.zeros(file_count)
qpie_entropy_val = np.zeros(file_count)

qpie_edge_density_QHED = np.zeros(file_count)
qpie_thickness_ratio_QHED = np.zeros(file_count)
qpie_edge_fragments_QHED = np.zeros(file_count)
qpie_entropy_val_QHED = np.zeros(file_count)

################################################
frqi_edge_density = np.zeros(file_count)
frqi_thickness_ratio = np.zeros(file_count)
frqi_edge_fragments = np.zeros(file_count)
frqi_entropy_val = np.zeros(file_count)

frqi_edge_density_QHED = np.zeros(file_count)
frqi_thickness_ratio_QHED = np.zeros(file_count)
frqi_edge_fragments_QHED = np.zeros(file_count)
frqi_entropy_val_QHED = np.zeros(file_count)
################################################

pixel_size = 512
# threshold = [0.225, 0.2, 0.15,0.225]
# threshold = [0.375, 0.2, 0.32, 0.15, 0.35] ### Animal Dataset (Eagle, Lion, Deer, Dog,Cat)
threshold = [0.125,0.2,0.125,0.1,0.2,0.25,0.13, 0.12] ### (Caterpillar,Duck,House,Elephant,Kitchen,Photographer,Lawn,HouseTree)

file_no = 7

i_file = 0
# Iterate over all files in the dataset
for entry in dataset_path.iterdir():

    # Optionally, filter to only process image files based on their extension
    if entry.is_file() and entry.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']:

        if i_file != file_no:
          i_file = i_file + 1
          continue

        print("\n Image File: ", entry.name)

        img_file = entry.name
        # img_name = img_file.split('.')[0]  # Split by '.' and take the first part
        # otherwise use entry.stem (name of the file without extension)

        image_path = dataset_path / img_file
        test_image = load_and_display_image(image_path,display_img = False)

        print("\n --------------------------------------------- \n")

        print("\n Image Size: ", pixel_size, "x", pixel_size)
        image = resize_image(test_image, (pixel_size,pixel_size))
        plot_image(image,entry.name)

        ##################### QPIE SOBEL Kernel #####################
        ########## Encoding ##########
        print("----- QPIE -----")
        print("\n Performing QPIE Sobel Gradient Encoding ----- ")

        start_time = time.perf_counter()

        print("\n Performing QPIE Sobel Gradient Encoding along x-direction ----- ")
        qc_qpie_x, qpie_grad_type = qpie_gradient(image,grad_type='lag2')

        print("\n Performing QPIE Sobel Gradient Encoding along y-direction ----- ")
        qc_qpie_y, qpie_grad_type = qpie_gradient(image.T,grad_type='lag2')
        # print(qc_QPIE.draw())

        end_time = time.perf_counter()
        elapsed_seconds = end_time - start_time
        print(f"QPIE Sobel Gradient Encoding Complete! Elapsed time: {elapsed_seconds:.6f} seconds\n")

        ##### Decoding
        print("\n Retreiving QPIE Sobel Gradient ----- ")

        start_time = time.perf_counter()

        qpie_statevector_x = run_simulation_statevector(qc_qpie_x,backend_sim)
        qpie_statevector_y = run_simulation_statevector(qc_qpie_y,backend_sim)

        # print(statevector_x)
        # print(statevector_y)

        qpie_grad_x = retrieve_grad_qpie(qpie_statevector_x,qpie_grad_type, Sobel = True)
        qpie_grad_y = retrieve_grad_qpie(qpie_statevector_y,qpie_grad_type, Sobel = True)

        end_time = time.perf_counter()
        elapsed_seconds = end_time - start_time
        print(f"QPIE Sobel Gradient Retrieval Complete! Elapsed time: {elapsed_seconds:.6f} seconds\n")

        ### Gradient Magnitude
        qpie_sobel_mag = np.sqrt(qpie_grad_x**2 + (qpie_grad_y.T)**2)

        ### Normalizing gradient
        qpie_sobel_mag = cv2.normalize(qpie_sobel_mag, None, 0, 1.0, cv2.NORM_MINMAX)

        # threshold = 0.3  # You can tune this value
        qpie_edges = (qpie_sobel_mag > threshold[i_file]).astype(np.uint8)
        # Making edge thickness = 1
        qpie_edges_thin = skeletonize((qpie_edges > 0).astype(np.uint8))

        ####################################################################################
        ####################################################################################
        ##################### QPIE QHED #####################
        ########## Encoding ##########
        print("\n Performing QPIE QHED Gradient Encoding ----- ")

        start_time = time.perf_counter()

        print("\n Performing QPIE QHED Gradient Encoding along x-direction ----- ")
        qc_qpie_x_QHED, qpie_grad_type_QHED = qpie_gradient(image,grad_type='lag1')

        print("\n Performing QPIE QHED Gradient Encoding along y-direction ----- ")
        qc_qpie_y_QHED, qpie_grad_type_QHED = qpie_gradient(image.T,grad_type='lag1')
        # print(qc_QPIE.draw())

        end_time = time.perf_counter()
        elapsed_seconds = end_time - start_time
        print(f"QPIE QHED Gradient Encoding Complete! Elapsed time: {elapsed_seconds:.6f} seconds\n")

        ##### Decoding
        print("\n Retreiving QPIE QHED Gradient ----- ")

        start_time = time.perf_counter()

        qpie_statevector_x_QHED = run_simulation_statevector(qc_qpie_x_QHED,backend_sim)
        qpie_statevector_y_QHED = run_simulation_statevector(qc_qpie_y_QHED,backend_sim)

        # print(statevector_x)
        # print(statevector_y)

        qpie_grad_x_QHED = retrieve_grad_qpie(qpie_statevector_x_QHED,qpie_grad_type_QHED)
        qpie_grad_y_QHED = retrieve_grad_qpie(qpie_statevector_y_QHED,qpie_grad_type_QHED)

        end_time = time.perf_counter()
        elapsed_seconds = end_time - start_time
        print(f"QPIE QHED Gradient Retrieval Complete! Elapsed time: {elapsed_seconds:.6f} seconds\n")

        ### Gradient Magnitude
        qpie_sobel_mag_QHED = np.sqrt(qpie_grad_x_QHED**2 + (qpie_grad_y_QHED.T)**2)

        ### Normalizing gradient
        qpie_sobel_mag_QHED = cv2.normalize(qpie_sobel_mag_QHED, None, 0, 1.0, cv2.NORM_MINMAX)

        # threshold = 0.3  # You can tune this value
        qpie_edges_QHED = (qpie_sobel_mag_QHED > threshold[i_file]).astype(np.uint8)
        # Making edge thickness = 1
        qpie_edges_thin_QHED = skeletonize((qpie_edges_QHED > 0).astype(np.uint8))

        ####################################################################################
        ####################################################################################

        print("----- FRQI -----")
        ##################### FRQI SOBEL Kernel #####################
        ########## Encoding ##########
        print("\n Performing FRQI Sobel Gradient Encoding ----- ")

        start_time = time.perf_counter()

        print("\n Performing FRQI Sobel Gradient Encoding along x-direction ----- ")
        qc_frqi_x, frqi_grad_type = frqi_gradient(image,grad_type='lag2')

        print("\n Performing FRQI Sobel Gradient Encoding along y-direction ----- ")
        qc_frqi_y, frqi_grad_type = frqi_gradient(image.T,grad_type='lag2')

        end_time = time.perf_counter()
        elapsed_seconds = end_time - start_time
        print(f"FRQI Sobel Gradient Encoding Complete! Elapsed time: {elapsed_seconds:.6f} seconds\n")

        ##### Decoding
        print("\n Retreiving FRQI Sobel Gradient ----- ")

        start_time = time.perf_counter()

        frqi_statevector_x = run_simulation_statevector(qc_frqi_x,backend_sim)
        frqi_statevector_y = run_simulation_statevector(qc_frqi_y,backend_sim)

        frqi_grad_x = retrieve_grad_frqi(frqi_statevector_x,frqi_grad_type, Sobel = True)
        frqi_grad_y = retrieve_grad_frqi(frqi_statevector_y,frqi_grad_type, Sobel = True)

        end_time = time.perf_counter()
        elapsed_seconds = end_time - start_time
        print(f"FRQI Sobel Gradient Retrieval Complete! Elapsed time: {elapsed_seconds:.6f} seconds\n")

        ### Gradient Magnitude
        frqi_sobel_mag = np.sqrt(frqi_grad_x**2 + (frqi_grad_y.T)**2)

        ### Normalizing gradient
        frqi_sobel_mag = cv2.normalize(frqi_sobel_mag, None, 0, 1.0, cv2.NORM_MINMAX)

        # threshold = 0.3  # You can tune this value
        frqi_edges = (frqi_sobel_mag > threshold[i_file]).astype(np.uint8)
        # Making edge thickness = 1
        frqi_edges_thin = skeletonize((frqi_edges > 0).astype(np.uint8))

        ####################################################################################
        ####################################################################################

        ##################### FRQI QHED #####################
        ########## Encoding ##########
        print("\n Performing FRQI QHED Gradient Encoding ----- ")

        start_time = time.perf_counter()

        print("\n Performing FRQI QHED Gradient Encoding along x-direction ----- ")
        qc_frqi_x_QHED, frqi_grad_type_QHED = frqi_gradient(image,grad_type='lag1')

        print("\n Performing FRQI QHED Gradient Encoding along y-direction ----- ")
        qc_frqi_y_QHED, frqi_grad_type_QHED = frqi_gradient(image.T,grad_type='lag1')
        # print(qc_FRQI.draw())

        end_time = time.perf_counter()
        elapsed_seconds = end_time - start_time
        print(f"FRQI QHED Gradient Encoding Complete! Elapsed time: {elapsed_seconds:.6f} seconds\n")

        ##### Decoding
        print("\n Retreiving FRQI QHED Gradient ----- ")

        start_time = time.perf_counter()

        frqi_statevector_x_QHED = run_simulation_statevector(qc_frqi_x_QHED,backend_sim)
        frqi_statevector_y_QHED = run_simulation_statevector(qc_frqi_y_QHED,backend_sim)

        frqi_grad_x_QHED = retrieve_grad_frqi(frqi_statevector_x_QHED,frqi_grad_type_QHED)
        frqi_grad_y_QHED = retrieve_grad_frqi(frqi_statevector_y_QHED,frqi_grad_type_QHED)

        end_time = time.perf_counter()
        elapsed_seconds = end_time - start_time
        print(f"FRQI QHED Gradient Retrieval Complete! Elapsed time: {elapsed_seconds:.6f} seconds\n")

        ### Gradient Magnitude
        frqi_sobel_mag_QHED = np.sqrt(frqi_grad_x_QHED**2 + (frqi_grad_y_QHED.T)**2)

        ### Normalizing gradient
        frqi_sobel_mag_QHED = cv2.normalize(frqi_sobel_mag_QHED, None, 0, 1.0, cv2.NORM_MINMAX)

        # threshold = 0.3  # You can tune this value
        frqi_edges_QHED = (frqi_sobel_mag_QHED > threshold[i_file]).astype(np.uint8)
        # Making edge thickness = 1
        frqi_edges_thin_QHED = skeletonize((frqi_edges_QHED > 0).astype(np.uint8))

        ####################################################################################
        ####################################################################################
        print("Edge Detection using Sobel Kernel")
        plot_images([image, qpie_edges, qpie_edges_thin],["Original image", "QPIE Sobel","Thin edges"])
        print("Edge Detection using QHED")
        plot_images([image, qpie_edges_QHED, qpie_edges_thin_QHED],["Original image", "QPIE QHED","Thin edges"])

        ####################################################################################
        ####################################################################################

        print("Edge Detection using Sobel Kernel")
        plot_images([image, frqi_edges, frqi_edges_thin],["Original image", "FRQI Sobel","Thin edges"])
        print("Edge Detection using QHED")
        plot_images([image, frqi_edges_QHED, frqi_edges_thin_QHED],["Original image", "FRQI QHED","Thin edges"])

        ####################################################################################
        ####################################################################################
        ### QPIE
        ### Calculating Performance Metrics Sobel Kernel
        qpie_edge_density[i_file] = np.sum(qpie_edges_thin) / qpie_edges_thin.size

        skeleton = skeletonize(qpie_edges)
        qpie_thickness_ratio[i_file] = np.sum(qpie_edges) / np.sum(skeleton)

        contours, _ = cv2.findContours(qpie_edges_thin.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        qpie_edge_fragments[i_file] = len(contours)

        qpie_entropy_val[i_file] = shannon_entropy(qpie_edges_thin)

        ### Calculating Performance Metrics QHED
        qpie_edge_density_QHED[i_file] = np.sum(qpie_edges_thin_QHED) / qpie_edges_thin.size

        skeleton = skeletonize(qpie_edges_QHED)
        qpie_thickness_ratio_QHED[i_file] = np.sum(qpie_edges_QHED) / np.sum(skeleton)

        contours, _ = cv2.findContours(qpie_edges_thin_QHED.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        qpie_edge_fragments_QHED[i_file] = len(contours)

        qpie_entropy_val_QHED[i_file] = shannon_entropy(qpie_edges_thin_QHED)

        ####################################################################################
        ####################################################################################
        ### FRQI
        ### Calculating Performance Metrics Sobel Kernel
        frqi_edge_density[i_file] = np.sum(frqi_edges_thin) / frqi_edges_thin.size

        skeleton = skeletonize(frqi_edges)
        frqi_thickness_ratio[i_file] = np.sum(frqi_edges) / np.sum(skeleton)

        contours, _ = cv2.findContours(frqi_edges_thin.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        frqi_edge_fragments[i_file] = len(contours)

        frqi_entropy_val[i_file] = shannon_entropy(frqi_edges_thin)


        ### Calculating Performance Metrics QHED
        frqi_edge_density_QHED[i_file] = np.sum(frqi_edges_thin_QHED) / frqi_edges_thin.size

        skeleton = skeletonize(frqi_edges_QHED)
        frqi_thickness_ratio_QHED[i_file] = np.sum(frqi_edges_QHED) / np.sum(skeleton)

        contours, _ = cv2.findContours(frqi_edges_thin_QHED.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        frqi_edge_fragments_QHED[i_file] = len(contours)

        frqi_entropy_val_QHED[i_file] = shannon_entropy(frqi_edges_thin_QHED)

        ####################################################################################
        ####################################################################################
        print("\n Performance Metrics (Sobel Kernel) ----- ")

        print(f"Edge Density: (FRQI) {frqi_edge_density[i_file]:.4f} (QPIE) {qpie_edge_density[i_file]:.4f}")
        print(f"Average Edge Thickness: (FRQI) {frqi_thickness_ratio[i_file]:.2f} (QPIE) {qpie_thickness_ratio[i_file]:.2f} ")
        print(f"Number of edge fragments: (FRQI) {frqi_edge_fragments[i_file]} (QPIE) {qpie_edge_fragments[i_file]} ")
        print(f"Edge Map Entropy: (FRQI) {frqi_entropy_val[i_file]:.2f} (QPIE) {qpie_entropy_val[i_file]:.2f}")

        print("\n Performance Metrics (QHED) ----- ")
        print(f"Edge Density: (FRQI) {frqi_edge_density_QHED[i_file]:.4f} (QPIE) {qpie_edge_density_QHED[i_file]:.4f}")
        print(f"Average Edge Thickness: (FRQI) {frqi_thickness_ratio_QHED[i_file]:.2f} (QPIE) {qpie_thickness_ratio_QHED[i_file]:.2f} ")
        print(f"Number of edge fragments: (FRQI) {frqi_edge_fragments_QHED[i_file]} (QPIE) {qpie_edge_fragments_QHED[i_file]} ")
        print(f"Edge Map Entropy: (FRQI) {frqi_entropy_val_QHED[i_file]:.2f} (QPIE) {qpie_entropy_val_QHED[i_file]:.2f}")

        ####################################################################################
        ####################################################################################

        ### Saving reconstructed image
        #### QPIE Saving ####
        # img_output_name = f"QPIE_{entry.stem}_Sobel{entry.suffix}"
        img_output_name = f"QPIE_{entry.stem}_Sobel.jpg"
        output_path = dataset_path.parent / 'Edge Detection using Sobel' / img_output_name
        save_image(qpie_edges, file_name = output_path)

        img_output_name = f"QPIE_{entry.stem}_SobelThin.jpg"
        output_path = dataset_path.parent / 'Edge Detection using Sobel' / img_output_name
        save_image(qpie_edges_thin, file_name = output_path)

        img_output_name = f"QPIE_{entry.stem}_QHED.jpg"
        output_path = dataset_path.parent / 'Edge Detection using QHED' / img_output_name
        save_image(qpie_edges_QHED, file_name = output_path)

        img_output_name = f"QPIE_{entry.stem}_QHEDThin.jpg"
        output_path = dataset_path.parent / 'Edge Detection using QHED' / img_output_name
        save_image(qpie_edges_thin_QHED, file_name = output_path)

        #### FRQI Saving ####
        img_output_name = f"FRQI_{entry.stem}_Sobel.jpg"
        output_path = dataset_path.parent / 'Edge Detection using Sobel' / img_output_name
        save_image(frqi_edges, file_name = output_path)

        img_output_name = f"FRQI_{entry.stem}_SobelThin.jpg"
        output_path = dataset_path.parent / 'Edge Detection using Sobel' / img_output_name
        save_image(frqi_edges_thin, file_name = output_path)

        img_output_name = f"FRQI_{entry.stem}_QHED.jpg"
        output_path = dataset_path.parent / 'Edge Detection using QHED' / img_output_name
        save_image(frqi_edges_QHED, file_name = output_path)

        img_output_name = f"FRQI_{entry.stem}_QHEDThin.jpg"
        output_path = dataset_path.parent / 'Edge Detection using QHED' / img_output_name
        save_image(frqi_edges_thin_QHED, file_name = output_path)

        i_file = i_file + 1

Output hidden; open in https://colab.research.google.com to view.

## Metric Evaluation

In [ ]:
# Save the arrays as .npy files in the output folder
np.save(dataset_path.parent / 'Edge Detection using Sobel' / 'Metric' / 'qpie_edge_density.npy', qpie_edge_density)
np.save(dataset_path.parent / 'Edge Detection using Sobel' / 'Metric' / 'qpie_edge_fragments.npy', qpie_edge_fragments)
np.save(dataset_path.parent / 'Edge Detection using Sobel' / 'Metric' / 'qpie_thickness_ratio.npy', qpie_thickness_ratio)
np.save(dataset_path.parent / 'Edge Detection using Sobel' / 'Metric' / 'qpie_entropy_val.npy', qpie_entropy_val)

np.save(dataset_path.parent / 'Edge Detection using QHED' / 'Metric' / 'qpie_edge_density_QHED.npy', qpie_edge_density_QHED)
np.save(dataset_path.parent / 'Edge Detection using QHED' / 'Metric' / 'qpie_edge_fragments_QHED.npy', qpie_edge_fragments_QHED)
np.save(dataset_path.parent / 'Edge Detection using QHED' / 'Metric' / 'qpie_thickness_ratio_QHED.npy', qpie_thickness_ratio_QHED)
np.save(dataset_path.parent / 'Edge Detection using QHED' / 'Metric' / 'qpie_entropy_val_QHED.npy', qpie_entropy_val_QHED)

np.save(dataset_path.parent / 'Edge Detection using Sobel' / 'Metric' / 'frqi_edge_density.npy', frqi_edge_density)
np.save(dataset_path.parent / 'Edge Detection using Sobel' / 'Metric' / 'frqi_edge_fragments.npy', frqi_edge_fragments)
np.save(dataset_path.parent / 'Edge Detection using Sobel' / 'Metric' / 'frqi_thickness_ratio.npy', frqi_thickness_ratio)
np.save(dataset_path.parent / 'Edge Detection using Sobel' / 'Metric' / 'frqi_entropy_val.npy', frqi_entropy_val)

np.save(dataset_path.parent / 'Edge Detection using QHED' / 'Metric' / 'frqi_edge_density_QHED.npy', frqi_edge_density_QHED)
np.save(dataset_path.parent / 'Edge Detection using QHED' / 'Metric' / 'frqi_edge_fragments_QHED.npy', frqi_edge_fragments_QHED)
np.save(dataset_path.parent / 'Edge Detection using QHED' / 'Metric' / 'frqi_thickness_ratio_QHED.npy', frqi_thickness_ratio_QHED)
np.save(dataset_path.parent / 'Edge Detection using QHED' / 'Metric' / 'frqi_entropy_val_QHED.npy', frqi_entropy_val_QHED)


print("Metrics saved successfully!")